# Opening Distribution in the FIDE Candidates Tournament

Comparing opening choices across the 2022, 2024, and 2026 Candidates Tournaments using data from Lichess broadcasts.

In [1]:
import io
import time
import chess.pgn
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Round IDs for each Candidates tournament (Open section), scraped from Lichess
TOURNAMENTS = {
    "2022": {
        "name": "FIDE Candidates Tournament 2022",
        "round_ids": [
            "LsFeKWZU", "sylFQGas", "oe2udItH", "0QuWnLkU", "1ZAF8srK",
            "yF4JxPcn", "OFBhwamI", "fsvj5GFW", "pk9gRSMB", "FWJYzJJJ",
            "16iH8elQ", "ZfmMC2Gh", "oIi8sTms", "ZA07lchF",
        ],
    },
    "2024": {
        "name": "FIDE Candidates 2024 (Open)",
        "round_ids": [
            "AjqSsU1w", "GenKIJ8A", "xQgaUu2y", "CPS9dENa", "MDiLWQ5M",
            "kPqEUBZJ", "X9LGGQoT", "nUycmG6L", "A7SWsX0x", "vfqUR38R",
            "46ohJ8Qt", "eJghIBZe", "Dc0DQMaQ", "S4zisI6M",
        ],
    },
    "2026": {
        "name": "FIDE Candidates 2026 (Open)",
        "round_ids": [
            "uLCZwqAK", "FRTlzP2X", "SDizieNR", "97di6JjX", "liBI9Brw",
            "WmbQrSNz", "aVbIuZ7Q", "XYRPyV8x", "hTVc2Qgj", "G3oSxPgs",
            "seMTFSPr", "9SHysZuu", "rFG1W5Tp", "oBkeHnpi",
        ],
    },
}

In [3]:
def fetch_round_pgn(round_id: str) -> str:
    """Fetch PGN text for a single broadcast round from Lichess."""
    url = f"https://lichess.org/api/broadcast/round/{round_id}.pgn"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text


def parse_games(pgn_text: str) -> list[dict]:
    """Parse all games from a PGN string and extract header info."""
    games = []
    pgn = io.StringIO(pgn_text)
    while True:
        game = chess.pgn.read_game(pgn)
        if game is None:
            break
        h = game.headers
        games.append({
            "white": h.get("White", ""),
            "black": h.get("Black", ""),
            "result": h.get("Result", ""),
            "eco": h.get("ECO", ""),
            "opening": h.get("Opening", ""),
            "date": h.get("Date", ""),
            "round": h.get("Round", ""),
        })
    return games

In [4]:
all_games = []

for year, info in TOURNAMENTS.items():
    print(f"Fetching {info['name']}...")
    for i, rid in enumerate(info["round_ids"], 1):
        pgn_text = fetch_round_pgn(rid)
        games = parse_games(pgn_text)
        for g in games:
            g["tournament"] = year
        all_games.extend(games)
        print(f"  Round {i}: {len(games)} games")
        time.sleep(0.5)  # be polite to the Lichess API

df = pd.DataFrame(all_games)
# Drop games with no opening annotation (unannotated/not yet started)
df = df[df["opening"] != "?"].reset_index(drop=True)
print(f"\nTotal games: {len(df)}")
df.head()

Fetching FIDE Candidates Tournament 2022...


  Round 1: 4 games


  Round 2: 4 games


  Round 3: 4 games


  Round 4: 4 games


...

  Round 12: 4 games


  Round 13: 4 games


  Round 14: 4 games



Total games: 168


,white,black,result,eco,opening,date,round,tournament
0,"Duda, Jan-Krzysztof","Rapport, Richard",1/2-1/2,B44,Sicilian Defense: Taimanov Variation,2022.06.17,1.1,2022
1,"Ding, Liren","Nepomniachtchi, Ian",0-1,A20,English Opening: King's English Variation,2022.06.17,1.2,2022
2,"Caruana, Fabiano","Nakamura, Hikaru",1-0,C65,"Ruy Lopez: Berlin Defense, Anti-Berlin Variation",2022.06.17,1.3,2022
3,"Radjabov, Teimour","Firouzja, Alireza",1/2-1/2,D24,Queen's Gambit Accepted,2022.06.17,1.4,2022
4,Richard Rapport,Alireza Firouzja,1/2-1/2,B53,Sicilian Defense: Chekhover Variation,2022.06.17,?,2022


In [5]:
df["eco_letter"] = df["eco"].str[0]
df["opening_family"] = df["opening"].str.split(":").str[0].str.split(",").str[0].str.strip()

print("Games per tournament:")
print(df.groupby("tournament").size())
print()
print("Sample of parsed fields:")
df[["tournament", "eco", "eco_letter", "opening", "opening_family"]].sample(10, random_state=42)

Games per tournament:
tournament
2022    56
2024    56
2026    56
dtype: int64

Sample of parsed fields:


,tournament,eco,eco_letter,opening,opening_family
137,2026,D38,D,Queen's Gambit Declined: Ragozin Defense,Queen's Gambit Declined
30,2022,C82,C,"Ruy Lopez: Open, Dilworth Variation",Ruy Lopez
119,2026,C42,C,Petrov's Defense: Karklins-Martinovsky Variation,Petrov's Defense
29,2022,C47,C,"Four Knights Game: Scotch Variation Accepted, ...",Four Knights Game
142,2026,D24,D,Queen's Gambit Accepted,Queen's Gambit Accepted
161,2026,D31,D,Queen's Gambit Declined: Charousek Variation,Queen's Gambit Declined
164,2026,B90,B,Sicilian Defense: Najdorf Variation,Sicilian Defense
51,2022,E04,E,Catalan Opening: Open Defense,Catalan Opening
105,2024,B30,B,Sicilian Defense: Nyezhmetdinov-Rossolimo Attack,Sicilian Defense
60,2024,D30,D,Queen's Gambit Declined,Queen's Gambit Declined


## Top Openings by Tournament

In [6]:
top_n = 10
top_families = df["opening_family"].value_counts().head(top_n).index

subset = df[df["opening_family"].isin(top_families)]
ct = subset.groupby(["tournament", "opening_family"]).size().unstack(fill_value=0)
ct = ct[top_families]

# Reshape for plotly: tournament on x-axis, opening as color
ct_plot = ct.reset_index().melt(id_vars="tournament", var_name="opening_family", value_name="count")

fig = px.bar(
    ct_plot,
    x="tournament",
    y="count",
    color="opening_family",
    barmode="group",
    title="Top Opening Families by Candidates Tournament",
    labels={"count": "Number of games", "tournament": "", "opening_family": ""},
)
fig.update_layout(legend_title_text="")
fig.show()

In [7]:
lines = ["| Opening Family | 2022 | 2024 | 2026 | Total |", "| :--- | :---: | :---: | :---: | :---: |"]
for fam in top_families:
    row = ct.loc[:, fam]
    total = row.sum()
    lines.append(f"| {fam} | {row.get('2022', 0)} | {row.get('2024', 0)} | {row.get('2026', 0)} | {total} |")

print("\n".join(lines))

| Opening Family | 2022 | 2024 | 2026 | Total |
| :--- | :---: | :---: | :---: | :---: |
| Ruy Lopez | 13 | 14 | 0 | 27 |
| Queen's Gambit Declined | 1 | 5 | 20 | 26 |
| Sicilian Defense | 11 | 9 | 4 | 24 |
| Petrov's Defense | 6 | 7 | 6 | 19 |
| English Opening | 4 | 0 | 8 | 12 |
| Italian Game | 5 | 5 | 1 | 11 |
| Catalan Opening | 5 | 1 | 3 | 9 |
| Nimzo-Indian Defense | 3 | 2 | 2 | 7 |
| Four Knights Game | 3 | 1 | 1 | 5 |
| French Defense | 0 | 3 | 2 | 5 |
